# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published Date: {metadata.date_published}\n")
print(f"License: {metadata.license}\n")
print(f"Version: {metadata.version}\n")

## 2. Data Overview
Review available record sets and fields, referencing them by their `@id`. This section will enumerate the main record sets and the field structures for selection.

In [ ]:
# List available record sets and their fields
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    print("")

# Save the IDs for use in later steps
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview to select which data to work with.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}\n")

# Let's select the first record set for further exploration
selected_record_set_id = record_set_ids[0]
print(f"Fields (@id) for selected record set: {selected_record_set_id}")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field for analysis (e.g., abundance or molecular weight)
# You may need to adjust these '@id's depending on the dataset overview above
numeric_fields = [
    col for col in dataframes[selected_record_set_id].columns
    if dataframes[selected_record_set_id][col].dtype in ['int64', 'float64']
]
if not numeric_fields:
    # Attempt to infer float columns by converting
    for col in dataframes[selected_record_set_id].columns:
        try:
            dataframes[selected_record_set_id][col] = pd.to_numeric(dataframes[selected_record_set_id][col])
            if dataframes[selected_record_set_id][col].dtype in ['int64', 'float64']:
                numeric_fields.append(col)
        except Exception:
            pass

if not numeric_fields:
    raise Exception('No numeric fields found for EDA.')

numeric_field_id = numeric_fields[0]
print(f"Using field: {numeric_field_id} for EDA analysis.")

# Basic threshold filtering
threshold = dataframes[selected_record_set_id][numeric_field_id].mean()
filtered_df = dataframes[selected_record_set_id][dataframes[selected_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a field if one exists (for demonstration, using the second column if possible)
group_fields = [col for col in dataframes[selected_record_set_id].columns if col != numeric_field_id]
if group_fields:
    group_field_id = group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No group field available to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(dataframes[selected_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If possible, relationship between numeric_field and group_field
if group_fields:
    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=filtered_df,
        x=group_field_id,
        y=numeric_field_id
    )
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded rich dataset metadata using the Croissant schema and `mlcroissant`.
- Enumerated the available record sets and their fields via unique `@id`s.
- Loaded table data from a record set into pandas DataFrames for analysis.
- Performed exploratory analysis on a numeric field, applying filtering, normalization, and (optionally) grouping.
- Created basic visualizations to understand the distribution and category-wise patterns in the dataset.

This workflow can be adapted or extended depending on which record sets or fields you wish to analyze. Further domain-specific or advanced machine learning tasks can build on this structured starting point.